# Statystyka dla Data Analyst/Engineer — Moduł 7: Podstawy testowania hipotez

To moduł-scharnier całego kursu. Do tej pory szacowaliśmy nieznane wartości (średnie,
proporcje) i podawaliśmy ich niepewność (przedziały ufności). Teraz nauczymy się
**podejmować decyzje** na podstawie danych: czy zaobserwowana różnica jest prawdziwym
efektem, czy tylko losowym szumem próbki? To pytanie brzmi prosto, ale wymaga precyzyjnego
aparatu pojęciowego, który zbudujemy w tym module — i którego będziemy używać w każdym
kolejnym teście aż do końca kursu.

## Spis treści
1. [Logika testowania hipotez: H0 i H1](#sec1)
2. [Wartość p (p-value)](#sec2)
3. [Poziom istotności i reguła decyzyjna](#sec3)
4. [Błędy I i II rodzaju, moc testu](#sec4)
5. [Pełny przykład: test dla jednej proporcji](#sec5)
6. [Intuicja wartości p przez symulację](#sec6)
7. [Podsumowanie i ćwiczenia](#sec7)


Każdy moduł tego kursu zaczyna się od wygenerowania tego samego zestawu danych syntetycznych -- dzięki ustalonemu ziarnu losowości (`np.random.default_rng(42)`) liczby zawsze wyjdą identyczne, więc możesz otworzyć dowolny moduł niezależnie od pozostałych, nawet po restarcie kernela.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
n = 400

miasto = rng.choice(
    ["Warszawa", "Krakow", "Gdansk", "Wroclaw", "Poznan"],
    size=n, p=[0.30, 0.25, 0.15, 0.15, 0.15],
)
region = rng.choice(["Polnoc", "Poludnie", "Wschod", "Zachod"], size=n)
wiek = rng.integers(18, 70, size=n)

# wydatki miesieczne: rozklad prawoskosny (typowy dla danych o wydatkach)
wydatki_miesieczne = rng.lognormal(mean=5.2, sigma=0.5, size=n).round(2)

# satysfakcja skorelowana z wydatkami + efekt regionu (celowo rozny) + szum losowy,
# przyciete do skali 1-10 -- dzieki temu Modul 10 (ANOVA) ma realna roznice miedzy grupami
efekt_regionu = {"Polnoc": 0.0, "Poludnie": -0.5, "Wschod": 0.3, "Zachod": 0.8}
satysfakcja = np.clip(
    wydatki_miesieczne / 45
    + np.array([efekt_regionu[r] for r in region])
    + rng.normal(0, 1.5, size=n),
    1, 10,
).round(1)

# grupa kampanii A/B -- z celowo wprowadzonym efektem w konwersji (grupa B lepsza)
grupa = rng.choice(["A", "B"], size=n)
prawdopodobienstwo_konwersji = np.where(grupa == "A", 0.10, 0.20)
konwersja = rng.binomial(1, prawdopodobienstwo_konwersji)

# wydatki przed/po wprowadzeniu programu lojalnosciowego (dane sparowane)
wydatki_przed = rng.normal(220, 40, size=n).round(2)
efekt_programu = rng.normal(15, 10, size=n)
wydatki_po = (wydatki_przed + efekt_programu).round(2)

klienci = pd.DataFrame({
    "klient_id": np.arange(1, n + 1),
    "miasto": miasto,
    "region": region,
    "wiek": wiek,
    "wydatki_miesieczne": wydatki_miesieczne,
    "satysfakcja": satysfakcja,
    "grupa_kampanii": grupa,
    "konwersja": konwersja,
    "wydatki_przed": wydatki_przed,
    "wydatki_po": wydatki_po,
})
klienci.head()


<a id="sec1"></a>
## 1. Logika testowania hipotez: H0 i H1

Testowanie hipotez działa trochę jak proces sądowy: zaczynamy od założenia "niewinności"
i szukamy dowodów wystarczająco mocnych, żeby je obalić.

- **Hipoteza zerowa (H0)** — status quo, "nic się nie dzieje", "nie ma różnicy/efektu".
  To hipoteza, którą *domyślnie zakładamy za prawdziwą*, dopóki dane nie dostarczą
  wystarczająco silnych dowodów przeciw niej.
- **Hipoteza alternatywna (H1 lub Ha)** — to, co próbujemy wykazać: że JEST różnica/
  efekt.

Kluczowe: test statystyczny **nigdy nie "dowodzi" H0**. Możemy tylko **odrzucić H0** (gdy
dowody są wystarczająco mocne) albo **nie odrzucić H0** (gdy dowodów jest za mało) — nigdy
nie mówimy "udowodniliśmy H0" ani "H0 jest prawdziwa", dokładnie jak w sądzie nie ma
werdyktu "udowodniona niewinność", tylko "winny" albo "niewinny wobec braku wystarczających
dowodów".

Przykład, który poprowadzi nas przez cały ten moduł: firma z branży e-commerce twierdzi,
że typowy współczynnik konwersji w tej branży to **12%**. Chcemy sprawdzić, czy nasz sklep
(cały zbiór `klienci`, traktowany teraz jako próbka z szerszej populacji sklepów) różni się
od tej wartości branżowej.

- H0: `p = 0.12` (nasz sklep nie różni się od średniej branżowej)
- H1: `p != 0.12` (nasz sklep różni się od średniej branżowej — w dowolną stronę)

In [ ]:
p_hat = klienci["konwersja"].mean()
n = len(klienci)
print(f"Obserwowany wspolczynnik konwersji: {p_hat:.3f} (n={n})")


<a id="sec2"></a>
## 2. Wartość p (p-value)

**Wartość p** to prawdopodobieństwo zaobserwowania wyniku *co najmniej tak skrajnego* jak
ten, który faktycznie otrzymaliśmy, **zakładając, że H0 jest prawdziwa**. Innymi słowy:
"gdyby prawda była taka jak mówi H0, jak bardzo zaskakujący/nietypowy byłby nasz wynik?".

- **Mała wartość p** (np. `0.01`) → nasz wynik byłby bardzo mało prawdopodobny, gdyby H0
  była prawdziwa → mocny dowód PRZECIW H0.
- **Duża wartość p** (np. `0.60`) → nasz wynik jest całkiem zwyczajny nawet przy
  założeniu, że H0 jest prawdziwa → brak wystarczających dowodów przeciw H0.

> ⚠️ **Najczęstsze błędne interpretacje wartości p**
>
> Wartość p **NIE JEST**: (1) prawdopodobieństwem, że H0 jest prawdziwa; (2) prawdopodobieństwem, że wynik powstał 'przez przypadek'; (3) miarą wielkości efektu (mała wartość p nie znaczy 'duży efekt' — może to być bardzo mały, nieistotny praktycznie efekt wykryty dzięki ogromnej próbce). Wartość p to WYŁĄCZNIE prawdopodobieństwo warunkowe: `P(dane co najmniej tak skrajne | H0 prawdziwa)`.

<a id="sec3"></a>
## 3. Poziom istotności i reguła decyzyjna

**Poziom istotności** (`alpha`, zwykle `0.05`) to próg, który ustalamy PRZED zobaczeniem
danych — jak małe musi być `p`, żebyśmy uznali dowód za "wystarczająco mocny". Reguła
decyzyjna jest prosta:

- jeśli `p <= alpha` → **odrzucamy H0** (wynik "istotny statystycznie"),
- jeśli `p > alpha` → **nie odrzucamy H0** (za mało dowodów, żeby ją obalić).

`alpha = 0.05` to konwencja (nie prawo natury!) — oznacza, że jesteśmy gotowi zaakceptować
5% ryzyko błędnego odrzucenia prawdziwej H0. W kontekstach o wyższej stawce (np. badania
medyczne) często używa się ostrzejszych progów (`0.01` albo mniej).

> 📊 **alpha ustalamy z góry, nie po fakcie**
>
> Ustalanie `alpha` PO zobaczeniu wartości p (np. "wyszło 0.07, to przyjmijmy alpha=0.10, żeby wyszło istotne") to poważny błąd metodologiczny, czasem nazywany *p-hacking*. Próg decyzyjny trzeba zadeklarować przed analizą, inaczej cała interpretacja traci sens.

<a id="sec4"></a>
## 4. Błędy I i II rodzaju, moc testu

Test statystyczny może się pomylić na dwa różne sposoby:

|                        | H0 prawdziwa                | H0 fałszywa                  |
|------------------------|------------------------------|-------------------------------|
| **Odrzucamy H0**       | Błąd I rodzaju (fałszywy alarm) | Poprawna decyzja            |
| **Nie odrzucamy H0**   | Poprawna decyzja             | Błąd II rodzaju (przeoczenie) |

- **Błąd I rodzaju** (fałszywy pozytyw) — odrzucamy H0, choć w rzeczywistości jest
  prawdziwa (np. "wykrywamy" efekt, którego nie ma). Prawdopodobieństwo tego błędu to
  dokładnie `alpha`.
- **Błąd II rodzaju** (fałszywy negatyw) — nie odrzucamy H0, choć w rzeczywistości jest
  fałszywa (np. przeoczamy prawdziwy efekt). Prawdopodobieństwo tego błędu oznaczamy
  `beta`.
- **Moc testu** (*power*) — prawdopodobieństwo POPRAWNEGO wykrycia efektu, gdy on
  faktycznie istnieje: `moc = 1 - beta`. Do mocy testu i planowania wielkości próby
  wrócimy szczegółowo w Module 14 (A/B testing).

Zmniejszenie `alpha` (żeby mieć mniej fałszywych alarmów) automatycznie **zwiększa**
ryzyko błędu II rodzaju (trudniej cokolwiek wykryć) — to fundamentalny kompromis, nie da
się poprawić obu jednocześnie bez zwiększenia wielkości próby.

<a id="sec5"></a>
## 5. Pełny przykład: test dla jednej proporcji

Wróćmy do naszego pytania: czy `p = 0.12` (H0) jest zgodne z zaobserwowanym `p_hat`?
Statystyka testowa dla testu jednej proporcji to z-score (poznany w Module 4):

```
z = (p_hat - p0) / sqrt(p0 * (1 - p0) / n)
```

gdzie `p0` to wartość z H0 (tu: `0.12`). Liczymy, ile odchyleń standardowych nasz
zaobserwowany wynik jest oddalony od wartości zakładanej przez H0, a potem zamieniamy `z`
na wartość p za pomocą CDF rozkładu normalnego.

In [ ]:
from scipy import stats

p0 = 0.12  # wartosc z H0
se_pod_h0 = (p0 * (1 - p0) / n) ** 0.5

z = (p_hat - p0) / se_pod_h0

# test dwustronny (H1: p != p0) -- mnozymy przez 2, bo interesuja nas oba "ogony"
wartosc_p = 2 * (1 - stats.norm.cdf(abs(z)))

print(f"z = {z:.3f}")
print(f"wartosc p = {wartosc_p:.4f}")

alpha = 0.05
if wartosc_p <= alpha:
    print(f"p <= alpha ({alpha}) -> odrzucamy H0")
else:
    print(f"p > alpha ({alpha}) -> nie odrzucamy H0")


`statsmodels` ma gotową funkcję do dokładnie tego testu (`proportions_ztest`), warto
znać jako skrót:

In [ ]:
from statsmodels.stats.proportion import proportions_ztest

liczba_sukcesow = klienci["konwersja"].sum()
z_sm, p_sm = proportions_ztest(count=liczba_sukcesow, nobs=n, value=p0)
print(f"z = {z_sm:.3f}, wartosc p = {p_sm:.4f}")


<a id="sec6"></a>
## 6. Intuicja wartości p przez symulację

Zamiast liczyć wartość p ze wzoru, możemy ją **oszacować symulacyjnie** — to znacznie
lepiej buduje intuicję, co ta liczba naprawdę oznacza. Symulujemy tysiące "wyimaginowanych
sklepów" ZAKŁADAJĄC, że H0 jest prawdziwa (`p = 0.12`), i sprawdzamy, jak często wynik
równie skrajny jak nasz `p_hat` pojawia się przez czysty przypadek.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
liczba_symulacji = 20_000

# symulujemy n prob Bernoulliego z p=0.12 (czyli: zakladajac H0), liczba_symulacji razy
symulowane_konwersje = rng.binomial(n=n, p=p0, size=liczba_symulacji) / n

# jak czesto symulowany wynik jest ROWNIE ALBO BARDZIEJ skrajny (w dowolna strone) niz nasz p_hat?
odchylenie_obserwowane = abs(p_hat - p0)
wartosc_p_symulowana = (np.abs(symulowane_konwersje - p0) >= odchylenie_obserwowane).mean()

print(f"Wartosc p z symulacji: {wartosc_p_symulowana:.4f}")
print(f"Wartosc p ze wzoru (sekcja 5): {wartosc_p:.4f}")

plt.figure(figsize=(8, 4))
plt.hist(symulowane_konwersje, bins=50, color="lightgray", edgecolor="white")
plt.axvline(p_hat, color="red", linestyle="--", label=f"Nasz obserwowany wynik: {p_hat:.3f}")
plt.axvline(p0, color="black", linestyle="-", label=f"H0: p = {p0}")
plt.xlabel("Symulowany wspolczynnik konwersji (zakladajac H0)")
plt.title("Rozklad wynikow pod H0 -- gdzie na tym tle jest nasz wynik?")
plt.legend()
plt.show()


Wartość p ze wzoru i z symulacji powinny wyjść bardzo blisko siebie — bo to dokładnie
to samo pytanie, tylko policzone dwoma metodami. Widać też wprost na wykresie: czerwona
linia (nasz wynik) leży wyraźnie na skraju "chmury" możliwych wyników pod H0 — to właśnie
znaczy mała wartość p.

<a id="sec7"></a>
## 7. Podsumowanie i ćwiczenia

W tym module poznaliśmy:
- logikę H0/H1 (założenie "niewinności", które próbujemy obalić dowodami),
- czym naprawdę jest (i nie jest) wartość p,
- poziom istotności `alpha` i regułę decyzyjną,
- błędy I i II rodzaju oraz moc testu,
- pełny, ręcznie policzony test dla jednej proporcji (z-test),
- symulacyjną intuicję stojącą za wartością p.

> 📝 **Ćwiczenie 1: Test dla innego benchmarku**
>
> Powtórz test z sekcji 5, ale dla `p0 = 0.15` zamiast `0.12`. Czy przy tym benchmarku nadal odrzucamy H0 na poziomie `alpha=0.05`?

<details>
<summary><b>👉 Kliknij, żeby zobaczyć przykładowe rozwiązanie</b></summary>

```python
p0_b = 0.15
se_b = (p0_b * (1 - p0_b) / n) ** 0.5
z_b = (p_hat - p0_b) / se_b
p_wartosc_b = 2 * (1 - stats.norm.cdf(abs(z_b)))
print(f"z={z_b:.3f}, p={p_wartosc_b:.4f}")
print("Odrzucamy H0" if p_wartosc_b <= 0.05 else "Nie odrzucamy H0")
```
</details>

> 📝 **Ćwiczenie 2: Test jednostronny**
>
> Firma twierdzi, że TYPOWA konwersja to CO NAJMNIEJ 12% (czyli H1 jest teraz jednostronna: `p < 0.12`, sprawdzamy czy jesteśmy PONIŻEJ benchmarku). Zmień wzór z sekcji 5 na test jednostronny: wartość p to teraz po prostu `stats.norm.cdf(z)` (bez mnożenia przez 2, i bez `abs()`). Policz ją i porównaj z wynikiem testu dwustronnego.

<details>
<summary><b>👉 Kliknij, żeby zobaczyć przykładowe rozwiązanie</b></summary>

```python
p_wartosc_jednostronna = stats.norm.cdf(z)
print(f"z={z:.3f}, p jednostronne={p_wartosc_jednostronna:.4f}")
print(f"(dla porownania, p dwustronne z sekcji 5: {wartosc_p:.4f})")
```

Test jednostronny daje mniejsza wartosc p przy tym samym z (bo caly 'zapas' istotnosci alpha idzie w jeden ogon) -- ale mozna go stosowac TYLKO gdy kierunek efektu zostal ustalony PRZED zobaczeniem danych, inaczej to kolejna forma p-hackingu.
</details>

> 📝 **Ćwiczenie 3: Symulacja dla innej wielkości próby**
>
> Powtórz symulację z sekcji 6, ale dla `n=50` zamiast `n=400` (przy tym samym obserwowanym `p_hat=0.185`, zaokrąglonym do najbliższej liczby całkowitej sukcesów). Czy przy mniejszej próbie ten sam obserwowany współczynnik nadal jest istotny statystycznie?

<details>
<summary><b>👉 Kliknij, żeby zobaczyć przykładowe rozwiązanie</b></summary>

```python
n_maly = 50
p_hat_maly = 0.185  # ten sam obserwowany wspolczynnik, mniejsza proba
sukcesy_maly = round(p_hat_maly * n_maly)

symulowane_male = rng.binomial(n=n_maly, p=p0, size=20_000) / n_maly
odchylenie_male = abs(sukcesy_maly / n_maly - p0)
p_wartosc_mala = (np.abs(symulowane_male - p0) >= odchylenie_male).mean()
print(f"Wartosc p przy n={n_maly}: {p_wartosc_mala:.4f}")
```

Przy mniejszej probie ten sam obserwowany wspolczynnik zazwyczaj daje WIEKSZA wartosc p (mniej dowodow) -- to bezposrednia ilustracja tego, dlaczego wielkosc proby ma znaczenie (Modul 5 i Modul 14).
</details>

> 🔥 **Wyzwanie: krzywa mocy testu**
>
> Napisz symulację sprawdzającą MOC testu (prawdopodobieństwo poprawnego odrzucenia H0, gdy PRAWDZIWY `p` faktycznie wynosi `0.16`, różniąc się od `p0=0.12` z H0). Dla `n=100`: zasymuluj 2000 'sklepów' z prawdziwym `p=0.16`, dla każdego policz wartość p testu (H0: `p0=0.12`), i policz odsetek przypadków, gdzie `p <= 0.05` (czyli test poprawnie wykrył różnicę). To właśnie jest przybliżona moc testu dla tego konkretnego efektu i tej wielkości próby.

<details>
<summary><b>👉 Kliknij, żeby zobaczyć przykładowe rozwiązanie</b></summary>

```python
prawdziwe_p = 0.16
n_mocy = 100
liczba_sklepow = 2000
wykrycia = 0

for _ in range(liczba_sklepow):
    sukcesy_sym = rng.binomial(n=n_mocy, p=prawdziwe_p)
    p_hat_sym = sukcesy_sym / n_mocy
    se_sym = (p0 * (1 - p0) / n_mocy) ** 0.5
    z_sym = (p_hat_sym - p0) / se_sym
    p_wartosc_sym = 2 * (1 - stats.norm.cdf(abs(z_sym)))
    if p_wartosc_sym <= 0.05:
        wykrycia += 1

print(f"Przyblizona moc testu: {wykrycia / liczba_sklepow:.3f}")
```

Wynik pokazuje, jak czesto test POPRAWNIE wykryje prawdziwa roznice o tej wielkosci przy n=100 -- moc ponizej ok. 0.80 zwykle uznaje sie za za niska w praktyce (wiecej o tym w Module 14).
</details>

## Co dalej?

Mamy już cały aparat pojęciowy: H0/H1, wartość p, alpha, błędy I/II rodzaju. W **Module 8**
zastosujemy go do najczęściej używanej rodziny testów — **testów t** — do porównywania
średnich: jednej próbki z wartością odniesienia, dwóch niezależnych grup, oraz danych
sparowanych (jak nasze `wydatki_przed`/`wydatki_po`).